In [ ]:
from pdmx import load
import os
from collections import defaultdict
import pickle

# =========================
#  CONFIG
# =========================

CONFIG = {
    "dataset_path": "/Volumes/HGST/PDMX",
    "grid": 1/6,
    "allowed_meters": {
        '4/4','2/4','3/4','3/8','6/8',
        '9/8','5/8','5/4','7/8','7/4','12/8'
    }
}

# =========================
# measure reconstruction utilities
# =========================

def get_measure_length(ts):
    return ts.numerator * (4 / ts.denominator)


EPS = 1e-6

def bucket_notes_into_measures(notes, measure_length):
    measures = defaultdict(list)
    for n in notes:
        idx = int((n.time + EPS) // measure_length)
        measures[idx].append(n)
    return measures


def get_measure_rhythm_from_notes(notes, measure_start, measure_length):
    notes = sorted(notes, key=lambda n: n.time)

    rhythm = []
    cursor = measure_start

    for n in notes:
        if n.time > cursor:
            rhythm.append(n.time - cursor)  # rest

        rhythm.append(n.duration)
        cursor = n.time + n.duration

    # trailing rest
    end = measure_start + measure_length
    if cursor < end:
        rhythm.append(end - cursor)

    return tuple(rhythm)


def has_spillover_notes(notes, measure_end):
    return any(n.time + n.duration > measure_end for n in notes)

# =========================
# FEATURE EXTRACTION
# =========================

def extract_rhythms_pdmx(music):
    rhythm_dict = defaultdict(lambda: defaultdict(int))

    if not music.tracks:
        return rhythm_dict

    track = music.tracks[0]

    if not music.time_signatures:
        return rhythm_dict

    ts = music.time_signatures[0]
    meter = f"{ts.numerator}/{ts.denominator}"

    measure_length = get_measure_length(ts)

    measures = bucket_notes_into_measures(track.notes, measure_length)
    measure_indices = sorted(measures.keys())

    for i, idx in enumerate(measure_indices):
        notes = measures[idx]

        start = idx * measure_length
        end = start + measure_length

        pattern = get_measure_rhythm_from_notes(notes, start, measure_length)

        if pattern:
            rhythm_dict[meter][pattern] += 1

        # spillover
        if has_spillover_notes(notes, end):
            if i + 1 < len(measure_indices):
                next_idx = measure_indices[i + 1]
                next_notes = measures[next_idx]

                next_pattern = get_measure_rhythm_from_notes(
                    next_notes,
                    next_idx * measure_length,
                    measure_length
                )

                combined = pattern + next_pattern
                rhythm_dict[meter][combined] += 1

    return rhythm_dict


def load_subset_paths(subset_file):
    with open(subset_file) as f:
        return [line.strip() for line in f if line.strip()]


def resolve_paths(base_path, relative_paths):
    return [os.path.join(base_path, p) for p in relative_paths]


def scan_dataset(path):
    global_dict = defaultdict(lambda: defaultdict(int))

    # 👇 NEW: load deduplicated subset
    subset_file = os.path.join(path, "subset_paths/deduplicated.txt")
    rel_paths = load_subset_paths(subset_file)
    paths = resolve_paths(path, rel_paths)

    for i, filepath in enumerate(paths):
        if not filepath.endswith(".json"):
            continue

        try:
            music = load(filepath)  # 👈 PDMX loader
        except Exception as e:
            print(f"Skipping {filepath}: {e}")
            continue

        local_dict = extract_rhythms_pdmx(music)

        # merge counts
        for meter, patterns in local_dict.items():
            for pat, count in patterns.items():
                global_dict[meter][pat] += count

        if (i + 1) % 1000 == 0:
            print(f"Processed {i+1} files")

    return global_dict


# =========================
#  CLEANING & NORMALIZATION
# =========================

def quantize(d, grid):
    return round(d / grid) * grid

def is_ternary_group(durs, beat=1.0, tol=1e-6):
    return (
        len(durs) == 3 and
        abs(sum(durs) - beat) < tol and
        all(abs(d - beat/3) < tol for d in durs)
    )

def get_beat_length(meter):
    return 1.5 if meter in ['6/8','9/8','12/8'] else 1.0


def clean_rhythm(pattern, grid, beat):
    q = [quantize(float(d), grid) for d in pattern]
    q = [d for d in q if d > 0]

    cleaned = []
    i = 0

    while i < len(q):
        # preserve ternary groups
        if i + 2 < len(q):
            group = q[i:i+3]
            if is_ternary_group(group, beat):
                cleaned.extend(group)
                i += 3
                continue

        d = q[i]

        # merge tiny durations
        if d < beat / 6 and i + 1 < len(q):
            q[i + 1] += d
        else:
            cleaned.append(d)

        i += 1

    return tuple(cleaned)


def clean_all(rhythm_dict, grid):
    cleaned_dict = defaultdict(lambda: defaultdict(int))

    for meter, patterns in rhythm_dict.items():
        beat = get_beat_length(meter)

        for pattern, count in patterns.items():
            cleaned = clean_rhythm(pattern, grid, beat)

            if cleaned:
                cleaned_dict[meter][cleaned] += count

    return cleaned_dict


# =========================
#  POSTPROCESSING
# =========================

def approx_equal(x, targets, tol=1e-6):
    return any(abs(x - t) < tol for t in targets)


def filter_patterns(rhythm_dict, allowed_meters):
    final_dict = {}

    for meter, patterns in rhythm_dict.items():

        if meter not in allowed_meters:
            continue

        final_dict[meter] = {}

        for pattern, count in patterns.items():
            pattern = tuple(x for x in pattern if x != 0)

            if not pattern:
                continue

            total = sum(pattern)

            # meter-specific validation
            if meter == '12/8' and not approx_equal(total, [6, 12]):
                continue
            if meter == '9/8' and not approx_equal(total, [4.5, 9]):
                continue

            final_dict[meter][pattern] = count

    return final_dict


# =========================
# PIPELINE
# =========================

def build_rhythm_dataset(config=CONFIG):
    print("Scanning dataset...")
    raw = scan_dataset(config["dataset_path"])  

    print("Cleaning rhythms...")
    cleaned = clean_all(raw, config["grid"])

    print("Filtering patterns...")
    final = filter_patterns(cleaned, config["allowed_meters"])

    print("Done.")
    return final


# =========================
# SAVE / LOAD
# =========================

def save_dataset(data, path="rhythm_dataset.pkl"):
    with open(path, "wb") as f:
        pickle.dump(data, f)

def load_dataset(path="rhythm_dataset.pkl"):
    with open(path, "rb") as f:
        return pickle.load(f)